# PM Assistant Demo

Minimal end-to-end demo for `llm_router_agent`.


In [ ]:
from dotenv import load_dotenv

from llm_router_agent import build_graph
from llm_router_agent.schemas import Team, TeamMember

load_dotenv()


In [ ]:
RUN_DEEP_REVIEW = False


In [ ]:
project_description = "Create a secure, multi-user AI dashboard using LangGraph and React."

team = Team(
    team_members=[
        TeamMember(name="Alice", role="Lead Dev", skills=["Python", "LangGraph", "Architecture"], seniority="Senior"),
        TeamMember(name="Bob", role="Frontend", skills=["React", "UI/UX"], seniority="Mid"),
        TeamMember(name="Charlie", role="QA", skills=["Testing", "Python"], seniority="Junior"),
    ]
)

initial_state = {
    "project_description": project_description,
    "team": team,
    "iteration_number": 0,
    "max_iteration": 2,
    "insights": [],
    "project_risk_score_iterations": [],
}


In [ ]:
graph = build_graph()
final_state = graph.invoke(initial_state, {"configurable": {"thread_id": "pm-assistant-demo"}})


In [ ]:
print("Tasks:")
for task in final_state["tasks"].task:
    print(f"- {task.id}: {task.task_name} ({task.estimated_day}d, skill={task.required_skill})")

print("\nDependencies:")
for dep in final_state.get("dependencies", []):
    print(f"- {dep.task_id} depends on {dep.dependent_on}")

print("\nSchedule:")
for item in final_state.get("schedule").schedule:
    print(f"- {item.task.task_name}: day {item.start_day} -> {item.end_day}")

print("\nAllocations:")
for alloc in final_state.get("task_allocations").task_allocations:
    print(f"- {alloc.task.task_name}: {alloc.team_member.name}")

print("\nRisks:")
for risk in final_state.get("risks").risks:
    print(f"- {risk.task_name}: {risk.score} ({risk.reason})")

print("\nInsights:")
for insight in final_state.get("insights", []):
    print(f"- {insight}")


In [ ]:
# Optional: Plotly Gantt (only if plotly + pandas are installed)
try:
    import pandas as pd
    import plotly.express as px

    rows = []
    alloc_by_task = {a.task.task_name: a.team_member.name for a in final_state["task_allocations"].task_allocations}
    for s in final_state["schedule"].schedule:
        rows.append({
            "Task": s.task.task_name,
            "Start": s.start_day,
            "Finish": s.end_day,
            "Assignee": alloc_by_task.get(s.task.task_name, "Unassigned"),
        })

    if rows:
        df = pd.DataFrame(rows)
        fig = px.timeline(df, x_start="Start", x_end="Finish", y="Task", color="Assignee", title="Project Schedule")
        fig.update_yaxes(autorange="reversed")
        fig.show()
    else:
        print("No schedule rows to plot.")
except Exception:
    print("Plotly/pandas not installed; skipping Gantt chart.")


In [ ]:
# Optional self-review toggle
if RUN_DEEP_REVIEW:
    print("Deep review enabled. Add your custom review prompt/model call here.")
